# Canasta SEPA - Mapa interactivo por sucursal

Este notebook procesa los datos del **Sistema Electrónico de Publicidad de Precios Argentinos (SEPA)** del Ministerio de Economía y construye el **Índice de Consumo Masivo UADE (ICM-UADE)** a nivel de sucursal individual.

A partir de los archivos mensuales del SEPA, calcula el valor de una canasta fija de 30 productos de consumo masivo para cada sucursal del país y genera un mapa interactivo HTML georreferenciado donde se visualiza el precio en cada punto de venta, con detalle de cadena, ubicación y composición de la canasta.

## Inputs requeridos en `/content/`

- `MMAAAA_pais_parte1COMPLETO.csv.gz` y `MMAAAA_pais_parte2COMPLETO.csv.gz` (datos SEPA del mes)
- `Maestro de Productos Interno.xlsx`
- `maestro_sucursales_completo.xlsx`

El notebook detecta automáticamente el mes a partir del nombre del archivo, por lo que sirve para cualquier período sin cambiar el código.

## Output

- `mapa_canasta_pais_MMAAAA_detalle.html` — mapa interactivo georreferenciado descargable

---

INECO — Instituto de Economía, UADE

In [1]:
# ============================================================
# BLOQUE 1 — Imports y configuración (parametrizable por mes)
# ============================================================
import os, re, gzip, gc
import pandas as pd
import numpy as np
import glob

# >>> CONFIGURACIÓN: detección automática del mes >>>
# Buscar archivos del SEPA en /content/ con patrón "MMAAAA_pais_parteN..."
patron_archivos = "/content/*_pais_parte*COMPLETO.csv.gz"
archivos_encontrados = sorted(glob.glob(patron_archivos))

if len(archivos_encontrados) == 0:
    raise RuntimeError(
        "❌ No se encontraron archivos SEPA en /content/.\n"
        "   Esperado: MMAAAA_pais_parteN_COMPLETO.csv.gz\n"
        "   Ej: 042026_pais_parte1COMPLETO.csv.gz"
    )

# Detectar el mes del primer archivo (extraer MMAAAA al inicio del nombre)
nombre_primer_archivo = os.path.basename(archivos_encontrados[0])
match = re.match(r'(\d{2})(\d{4})_pais_parte', nombre_primer_archivo)

if not match:
    raise RuntimeError(
        f"❌ No se pudo detectar el mes del archivo: {nombre_primer_archivo}\n"
        "   Esperado: MMAAAA_pais_parteN_COMPLETO.csv.gz"
    )

MES_NUM = match.group(1)        # "04", "05", "06", etc.
ANIO = match.group(2)            # "2026"
MES = f"{MES_NUM}{ANIO}"         # "042026"

# Nombre legible del mes en español
NOMBRES_MES = {
    '01': 'enero',    '02': 'febrero', '03': 'marzo',
    '04': 'abril',    '05': 'mayo',    '06': 'junio',
    '07': 'julio',    '08': 'agosto',  '09': 'septiembre',
    '10': 'octubre',  '11': 'noviembre', '12': 'diciembre',
}
NOMBRE_MES = f"{NOMBRES_MES[MES_NUM]} {ANIO}"     # "abril 2026"
NOMBRE_MES_TITLE = NOMBRE_MES.title()              # "Abril 2026"

# Archivos a procesar (todos los que coincidan con el patrón del mes)
patron_mes = f"/content/{MES}_pais_parte*COMPLETO.csv.gz"
ARCHIVOS_MES = sorted(glob.glob(patron_mes))

PATH_MAESTRO_PROD = "/content/Maestro de Productos Interno.xlsx"
PATH_MAESTRO_SUC = "/content/maestro_sucursales_completo.xlsx"

# Verificación
print(f"📅 Mes detectado: {NOMBRE_MES_TITLE} (parámetro MES = {MES})\n")
print(f"Archivos a procesar:")
for archivo in ARCHIVOS_MES:
    if os.path.exists(archivo):
        size_mb = os.path.getsize(archivo) / 1024 / 1024
        print(f"  ✅ {archivo}  ({size_mb:.1f} MB)")

print(f"\nMaestros:")
for path in [PATH_MAESTRO_PROD, PATH_MAESTRO_SUC]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  ✅ {path}  ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ NO ENCONTRADO: {path}")

📅 Mes detectado: Marzo 2026 (parámetro MES = 032026)

Archivos a procesar:
  ✅ /content/032026_pais_parte1COMPLETO.csv.gz  (154.2 MB)
  ✅ /content/032026_pais_parte2COMPLETO.csv.gz  (147.8 MB)

Maestros:
  ✅ /content/Maestro de Productos Interno.xlsx  (20.3 MB)
  ✅ /content/maestro_sucursales_completo.xlsx  (0.5 MB)


In [ ]:
# ============================================================
# BLOQUE 2 — Definición de canasta y carga de maestros
# ============================================================
CANASTA = {
    '7790742363008': ('Leche entera 1L',         20, 'Lácteos'),
    '7791337007628': ('Yogur 190g',               8, 'Lácteos'),
    '7791337061361': ('Queso Casancrem 290g',     2, 'Lácteos'),
    '7793940052002': ('Manteca 100g',             2, 'Lácteos'),
    '7791337007253': ('Cindor 1L',                4, 'Lácteos'),
    '7790272001029': ('Aceite girasol 1,5L',      2, 'Almacén'),
    '7790070433114': ('Arroz 500g',               2, 'Almacén'),
    '7790070320285': ('Fideos 500g',              4, 'Almacén'),
    '7792180140708': ('Harina leudante 1kg',      2, 'Almacén'),
    '7792710000182': ('Yerba 500g',               2, 'Almacén'),
    '7790550000157': ('Café 250g',                1, 'Almacén'),
    '7790040143234': ('Chocolinas 250g',          4, 'Almacén'),
    '7790072002080': ('Sal fina 500g',            1, 'Almacén'),
    '7790895000232': ('Coca Cola lata',           8, 'Bebidas'),
    '7790895067570': ('Coca Sin Azúcar 2,25L',    4, 'Bebidas'),
    '7798062548716': ('Agua Levite 500ml',        8, 'Bebidas'),
    '7793147118860': ('Cerveza lata',             6, 'Bebidas'),
    '7798074864675': ('Vino Malbec 750ml',        2, 'Bebidas'),
    '7790132098459': ('Lavandina 1L',             2, 'Limpieza'),
    '7791290794054': ('Detergente 300ml',         2, 'Limpieza'),
    '7793253003500': ('Limpiador Poett 900ml',    2, 'Limpieza'),
    '7791293047447': ('Shampoo 400ml',            1, 'Higiene'),
    '7791293045948': ('Acondicionador 340ml',     1, 'Higiene'),
    '7791293051208': ('Jabón tocador 90g',        4, 'Higiene'),
    '7791293049557': ('Antitranspirante',         2, 'Higiene'),
    '7891024183083': ('Hilo dental',              1, 'Higiene'),
    '7790770601899': ('Toallas femeninas x16',    2, 'Higiene'),
    '7790250015840': ('Papel higiénico',          2, 'Higiene'),
    '7790580327415': ('Rocklets 40g',             2, 'Snacks'),
    '7790580716707': ('Saladix 100g',             2, 'Snacks'),
}

CANASTA_EANS_LSTRIP = {e.lstrip('0') for e in CANASTA.keys()}
print(f"✅ Canasta definida: {len(CANASTA)} productos\n")

# Cargar maestros
print("Cargando maestros...")
maestro_prod = pd.read_excel(PATH_MAESTRO_PROD)
maestro_suc = pd.read_excel(PATH_MAESTRO_SUC)
print(f"  Productos:  {len(maestro_prod):,} filas")
print(f"  Sucursales: {len(maestro_suc):,} filas")

✅ Canasta definida: 30 productos

Cargando maestros...


In [ ]:
# ============================================================
# BLOQUE 3 (NACIONAL) — Procesar CSVs y filtrar canasta
# ============================================================
# Filtrar maestro de sucursales con coordenadas válidas dentro de Argentina
suc_pais = maestro_suc[
    maestro_suc['sucursales_latitud'].notna() &
    maestro_suc['sucursales_longitud'].notna() &
    (maestro_suc['sucursales_latitud'].between(-55, -22)) &
    (maestro_suc['sucursales_longitud'].between(-73, -53))
].copy()
print(f"Sucursales en todo el país con coordenadas válidas: {len(suc_pais):,}")

# IDs a string
suc_pais['id_comercio'] = suc_pais['id_comercio'].astype(str)
suc_pais['id_bandera'] = suc_pais['id_bandera'].astype(str)
suc_pais['id_sucursal'] = suc_pais['id_sucursal'].astype(str)

ids_pais = set(zip(suc_pais['id_comercio'],
                   suc_pais['id_bandera'],
                   suc_pais['id_sucursal']))
print(f"IDs únicos para matchear: {len(ids_pais):,}\n")

def normalizar_ean(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lstrip('0')
    return s if s else '0'

patron_fecha = re.compile(r'^precio_(\d{8})$')
acumulador = []

for archivo in ARCHIVOS_MES:
    nombre = os.path.basename(archivo)
    print(f"Procesando {nombre}...")

    df = pd.read_csv(archivo, compression='gzip', sep=',', dtype=str)
    print(f"  Total filas: {len(df):,}")

    cols_precio = [c for c in df.columns if patron_fecha.match(c)]
    print(f"  Columnas de precio detectadas: {len(cols_precio)} (días)")
    if cols_precio:
        print(f"  Rango: {cols_precio[0]} a {cols_precio[-1]}")

    # Filtrar por EAN de la canasta
    df['ean_norm'] = df['id_producto'].apply(normalizar_ean)
    df = df[df['ean_norm'].isin(CANASTA_EANS_LSTRIP)].copy()
    print(f"  Filas de canasta: {len(df):,}")

    # Filtrar por sucursales del país
    df['id_comercio'] = df['id_comercio'].astype(str)
    df['id_bandera'] = df['id_bandera'].astype(str)
    df['id_sucursal'] = df['id_sucursal'].astype(str)

    df['key_suc'] = list(zip(df['id_comercio'], df['id_bandera'], df['id_sucursal']))
    df = df[df['key_suc'].isin(ids_pais)].copy()
    df.drop(columns=['key_suc'], inplace=True)
    print(f"  Filas país + canasta: {len(df):,}")

    if len(df) == 0:
        continue

    # Wide → long
    cols_id = ['id_comercio','id_bandera','id_sucursal','ean_norm']
    df_long = df.melt(
        id_vars=cols_id,
        value_vars=cols_precio,
        var_name='fecha_col',
        value_name='precio_raw'
    )

    df_long['fecha'] = pd.to_datetime(
        df_long['fecha_col'].str.replace('precio_', '', regex=False),
        format='%Y%m%d'
    )
    df_long.drop(columns=['fecha_col'], inplace=True)

    df_long['precio'] = pd.to_numeric(df_long['precio_raw'], errors='coerce')
    df_long.drop(columns=['precio_raw'], inplace=True)

    df_long = df_long[df_long['precio'].notna() & (df_long['precio'] > 5)]
    print(f"  Observaciones diarias válidas: {len(df_long):,}\n")

    acumulador.append(df_long)
    del df, df_long
    gc.collect()

if not acumulador:
    raise RuntimeError("No hay datos")

datos = pd.concat(acumulador, ignore_index=True)
del acumulador; gc.collect()

# Eliminar duplicados
antes = len(datos)
datos = datos.drop_duplicates(
    subset=['fecha','id_comercio','id_bandera','id_sucursal','ean_norm'],
    keep='first'
)
duplicados = antes - len(datos)
if duplicados > 0:
    print(f"⚠️ Se eliminaron {duplicados:,} duplicados")

# Detectar factor de precio
EANS_REFERENCIA = ['7790072002080', '7790070320285', '7790132098459']
eans_ref_norm = {e.lstrip('0') for e in EANS_REFERENCIA}
ref = datos[datos['ean_norm'].isin(eans_ref_norm)]
mediana_ref = ref['precio'].median() if len(ref) else 0

if 30 <= mediana_ref <= 5000:
    FACTOR = 1
elif 3000 <= mediana_ref <= 500000:
    FACTOR = 100
elif mediana_ref > 500000:
    FACTOR = 10000
else:
    FACTOR = 1

datos['precio'] = datos['precio'] / FACTOR
print(f"Mediana de referencia: {mediana_ref:.2f} → Factor: {FACTOR}")

print(f"\n✅ Total observaciones diarias: {len(datos):,}")
print(f"Rango: {datos['fecha'].min().date()} → {datos['fecha'].max().date()}")
print(f"Sucursales únicas con datos: {datos.groupby(['id_comercio','id_bandera','id_sucursal']).ngroups:,}")

print(f"\nPrecio mediano por producto de referencia:")
for ean_str, nombre_prod in [('7790072002080','Sal Celusal'),
                              ('7790070320285','Fideos Favorita'),
                              ('7790132098459','Lavandina Ayudín')]:
    ean = ean_str.lstrip('0')
    sub = datos[datos['ean_norm'] == ean]
    if len(sub) > 0:
        print(f"  {nombre_prod}: ${sub['precio'].median():,.2f}")

In [ ]:
# ============================================================
# BLOQUE 4 — Promedio mensual por sucursal → canasta con detalle
# ============================================================
print("Calculando precio promedio mensual por sucursal-producto...")
precio_mes = (
    datos.groupby(['id_comercio','id_bandera','id_sucursal','ean_norm'])
    ['precio'].mean()
    .reset_index()
)
print(f"  Filas: {len(precio_mes):,}")

# Filtrar cadenas no representativas
CADENAS_FILTRAR = ['19', '2013', '3001', '4']
precio_mes = precio_mes[~precio_mes['id_comercio'].isin(CADENAS_FILTRAR)].copy()
print(f"  Filas después de filtrar cadenas no representativas: {len(precio_mes):,}")

# Distribución de productos por sucursal
productos_por_suc = (
    precio_mes.groupby(['id_comercio','id_bandera','id_sucursal'])
    ['ean_norm'].nunique()
    .reset_index(name='productos_propios')
)
print(f"\nDistribución de productos por sucursal:")
print(productos_por_suc['productos_propios'].describe())

# Precio promedio nacional por producto (para imputación)
precio_promedio_nacional = precio_mes.groupby('ean_norm')['precio'].mean().to_dict()

# Calcular canasta completa por sucursal CON DETALLE
print("\nCalculando canasta completa por sucursal con detalle de productos...")

def calcular_canasta_completa(grupo):
    productos_locales = dict(zip(grupo['ean_norm'], grupo['precio']))
    total = 0
    productos_propios = 0
    detalle = []

    for ean_raw, (nombre, qty, cat) in CANASTA.items():
        ean = ean_raw.lstrip('0')
        if ean in productos_locales:
            precio = productos_locales[ean]
            es_propio = True
            productos_propios += 1
        else:
            precio = precio_promedio_nacional.get(ean, 0)
            es_propio = False
        subtotal = precio * qty
        total += subtotal
        detalle.append((nombre, cat, qty, precio, subtotal, es_propio))

    return pd.Series({
        'canasta_total': total,
        'productos_propios': productos_propios,
        'detalle_productos': detalle
    })

canasta_sucursal = (
    precio_mes.groupby(['id_comercio','id_bandera','id_sucursal'])
    .apply(calcular_canasta_completa, include_groups=False)
    .reset_index()
)

# Filtrar sucursales con al menos 20 productos propios
MIN_PRODUCTOS = 20
canasta_sucursal = canasta_sucursal[
    canasta_sucursal['productos_propios'] >= MIN_PRODUCTOS
].copy()
print(f"\n✅ Sucursales con canasta válida (≥{MIN_PRODUCTOS} productos): {len(canasta_sucursal):,}")

# Cruzar con maestro
canasta_geo = canasta_sucursal.merge(
    suc_pais[['id_comercio','id_bandera','id_sucursal',
              'sucursales_nombre','sucursales_latitud','sucursales_longitud',
              'sucursales_barrio','sucursales_localidad','PROVINCIA','REGION']],
    on=['id_comercio','id_bandera','id_sucursal'],
    how='inner'
)

# MAPEO DE CADENAS VERIFICADO
nombres_cadenas_compuestas = {
    ('9', '1'):  'Vea',
    ('9', '2'):  'Disco',
    ('9', '3'):  'Jumbo',
    ('10', '1'): 'Carrefour',
    ('10', '2'): 'Carrefour Market',
    ('10', '3'): 'Carrefour Express',
    ('11', '2'): 'ChangoMas',
    ('11', '4'): 'Hiper ChangoMas',
    ('11', '5'): 'Mi ChangoMas',
    ('16', '1'): 'Hipermercado Libertad',
    ('16', '2'): 'Mini Libertad',
}

nombres_cadenas_simples = {
    '2':    'La Anónima',
    '3':    'Cadena 3',
    '5':    'Hipermercado Misiones',
    '8':    'Cadena 8 (Córdoba)',
    '12':   'Coto',
    '13':   'Cooperativa Obrera',
    '15':   'DIA',
    '20':   'LAR',
    '21':   'Toledo',
    '23':   'Cadena 23',
    '47':   'Pasamonte',
}

def asignar_cadena(row):
    key = (row['id_comercio'], row['id_bandera'])
    if key in nombres_cadenas_compuestas:
        return nombres_cadenas_compuestas[key]
    if row['id_comercio'] in nombres_cadenas_simples:
        return nombres_cadenas_simples[row['id_comercio']]
    return f"Cadena {row['id_comercio']}"

canasta_geo['cadena'] = canasta_geo.apply(asignar_cadena, axis=1)

print(f"\n✅ Canasta geolocalizada: {len(canasta_geo):,} sucursales")
print(f"Rango de precios: ${canasta_geo['canasta_total'].min():,.0f} a ${canasta_geo['canasta_total'].max():,.0f}")
print(f"Promedio país: ${canasta_geo['canasta_total'].mean():,.0f}")
print(f"Mediana país:  ${canasta_geo['canasta_total'].median():,.0f}")

print(f"\nDistribución por cadena:")
print(canasta_geo['cadena'].value_counts())

In [ ]:
# ============================================================
# BLOQUE 5 — Mapa interactivo con detalle de productos
# ============================================================
!pip install folium --quiet

import folium
from branca.colormap import LinearColormap

print(f"Construyendo mapa interactivo nacional para {NOMBRE_MES_TITLE}...")

centro_pais = [-38.0, -63.5]

m = folium.Map(
    location=centro_pais,
    zoom_start=5,
    tiles='cartodbpositron',
    control_scale=True
)

# Etiqueta "Islas Malvinas"
folium.map.Marker(
    location=[-51.7963, -59.5236],
    icon=folium.DivIcon(
        icon_size=(140, 28),
        icon_anchor=(70, 14),
        html='''
        <div style="
            background-color: rgba(255,255,255,0.95);
            border: 1px solid #777;
            border-radius: 3px;
            padding: 3px 7px;
            font-family: Arial, sans-serif;
            font-size: 11px;
            font-weight: 600;
            color: #222;
            text-align: center;
            white-space: nowrap;
        ">Islas Malvinas (ARG)</div>
        '''
    )
).add_to(m)

# Escala de colores
vmin = canasta_geo['canasta_total'].quantile(0.05)
vmax = canasta_geo['canasta_total'].quantile(0.95)

colormap = LinearColormap(
    colors=['#1a9850', '#66bd63', '#a6d96a', '#fee08b', '#fdae61', '#f46d43', '#d73027'],
    vmin=vmin,
    vmax=vmax,
    caption=f'Canasta promedio {NOMBRE_MES} (ARS)'
)
colormap.add_to(m)

# Funciones auxiliares
def fmt_ar(x):
    return f"{x:,.0f}".replace(",", ".")

def construir_tabla_detalle(detalle):
    cats = {}
    for nombre, cat, qty, precio, subtotal, es_propio in detalle:
        cats.setdefault(cat, []).append((nombre, qty, precio, subtotal, es_propio))

    filas = []
    for cat, items in cats.items():
        filas.append(f'<tr style="background:#0055A4;color:white;"><td colspan="4" style="padding:3px 5px;font-weight:bold;">{cat}</td></tr>')
        for nombre, qty, precio, subtotal, es_propio in items:
            estilo_fila = '' if es_propio else 'color:#888;font-style:italic;'
            marca = '' if es_propio else ' *'
            filas.append(
                f'<tr style="{estilo_fila}">'
                f'<td style="padding:2px 5px;">{nombre}{marca}</td>'
                f'<td style="padding:2px 5px;text-align:center;">x{qty}</td>'
                f'<td style="padding:2px 5px;text-align:right;">${fmt_ar(precio)}</td>'
                f'<td style="padding:2px 5px;text-align:right;font-weight:600;">${fmt_ar(subtotal)}</td>'
                f'</tr>'
            )

    return f'''
    <table style="width:100%;border-collapse:collapse;font-size:10px;font-family:Arial;">
        <thead>
            <tr style="background:#e6eef7;font-weight:bold;">
                <th style="padding:3px 5px;text-align:left;">Producto</th>
                <th style="padding:3px 5px;text-align:center;">Cant.</th>
                <th style="padding:3px 5px;text-align:right;">P.Unit</th>
                <th style="padding:3px 5px;text-align:right;">Subtotal</th>
            </tr>
        </thead>
        <tbody>
            {''.join(filas)}
        </tbody>
    </table>
    <div style="font-size:9px;color:#666;margin-top:4px;">* Producto no disponible en la sucursal; precio imputado con promedio nacional.</div>
    '''

# Agregar círculo por cada sucursal
for _, row in canasta_geo.iterrows():
    valor = row['canasta_total']
    color = colormap(valor)
    precio_fmt = fmt_ar(valor)

    tooltip_text = f"<b>{row['cadena']}</b><br>${precio_fmt}"
    tabla_html = construir_tabla_detalle(row['detalle_productos'])

    popup_html = f"""
    <div style="font-family: Arial; font-size: 12px; width: 420px; max-height: 500px; overflow-y: auto;">
        <h4 style="margin: 0; color: #0055A4;">{row['cadena']}</h4>
        <div style="font-size: 11px; color: #555; margin-bottom: 5px;">
            <b>{row['sucursales_nombre']}</b><br>
            {row.get('sucursales_barrio') or row.get('sucursales_localidad') or 'N/D'} — {row['PROVINCIA']}
        </div>
        <hr style="margin: 5px 0;">
        <div style="text-align: center; margin: 8px 0;">
            <span style="font-size: 11px; color: #666;">Canasta total</span><br>
            <span style="color: #0055A4; font-size: 20px; font-weight: bold;">${precio_fmt}</span><br>
            <span style="font-size: 10px; color: #888;">({row['productos_propios']}/30 productos propios)</span>
        </div>
        <hr style="margin: 5px 0;">
        {tabla_html}
    </div>
    """

    folium.CircleMarker(
        location=[row['sucursales_latitud'], row['sucursales_longitud']],
        radius=5,
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.8,
        weight=1,
        tooltip=tooltip_text,
        popup=folium.Popup(popup_html, max_width=450)
    ).add_to(m)

# Título dinámico
title_html = f'''
<div style="position: fixed;
            top: 10px; left: 50px; width: 400px; height: 60px;
            background-color: white; border: 2px solid #0055A4;
            border-radius: 5px; padding: 10px;
            font-family: Arial; z-index: 9999;">
    <b style="color: #0055A4; font-size: 14px;">ICM-UADE por sucursal - {NOMBRE_MES_TITLE}</b><br>
    <span style="font-size: 11px;">Todas las sucursales del país - click en cada punto para ver el detalle</span>
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

# Guardar con nombre dinámico
output_html = f'/content/mapa_canasta_pais_{MES}_detalle.html'
m.save(output_html)

print(f"\n✅ Mapa guardado en: {output_html}")
print(f"   Sucursales mostradas: {len(canasta_geo):,}")

from google.colab import files
files.download(output_html)

m